# Challenge 2 · Fluid Flow Around Fixed Obstacles

[Start Here](../../Start_Here.ipynb) · Previous: [Wave](../01_wave/Challenge_1_Wave_Dynamics.ipynb) · Next: [Climate](../03_climate/Challenge_3_Climate_Modeling.ipynb)

Predict velocity and pressure around fixed blocks in a channel. Solve steady incompressible Navier–Stokes flow around one block, repeat with three blocks, then return to one block for time-dependent flow. Check mass and momentum conservation alongside wall conditions and total flow rate.

### Run this Challenge

1. Set `USE_REFERENCE = False` in setup and run it. `True` runs the instructor solution, ignoring your edits. Check the printed mode.
2. Complete `student_equations` in the linked `.py` file. Save with Ctrl+S / Command+S; editing an example in this notebook does not change the program.
3. Run the level's training cell, then its result cell. Each attempt gets a fresh directory. After changing code, settings or mode, rerun both cells.
4. Set `STEPS = 2` and `DEVICE = "cpu"` for a quick execution check. Choose longer runs using the held-out errors and predictions below.

The code uses PhysicsNeMo 2.2.2: a `FullyConnected` network, a SymPy `PDE`, `PhysicsInformer`, and a PyTorch optimizer.

### Comparing attempts

First implement the equations, then change one training choice at a time. Keep the geometry, viscosity, density, inlet profile, startup ramp, and evaluation code fixed. Record your change, step count, seed, and errors so you can explain what improved.

Training uses your `student_equations`; held-out and unweighted PDE checks use the provided reference equations in both modes. Lower errors indicate improvement on those checks. These editable local results are practice feedback, not official scores or rankings. See [Assessment and feedback](../../ETC/course_materials/ASSESSMENT.md) for metric definitions.


In [ ]:
from pathlib import Path
from uuid import uuid4
import json
import os
import subprocess
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "02_challenges" / "02_fluid" / "chip_2d_l1.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open the notebook from inside the repository.")
sys.path.insert(0, str(ROOT))
from ETC.runtime.notebook import show_results, validate_settings

LAB_DIR = ROOT / "02_challenges" / "02_fluid"
USE_REFERENCE = os.environ.get("AI4SCI_REFERENCE", "0").lower() in {"1", "true", "yes"}
# USE_REFERENCE = False  # Uncomment to override the server default for student exercises.
DEVICE = os.environ.get("AI4SCI_DEVICE", "auto")
STEPS = int(os.environ.get("AI4SCI_STEPS", "200"))  # Verify runtime and accuracy on the event GPU.
SEED = 42
OUTPUT_BASE = Path(os.environ.get("AI4SCI_OUTPUT_DIR", str(LAB_DIR / "outputs"))).expanduser().resolve()
RUN_DIRS = {}  # Latest attempt per level; results are never shared between levels.
RUN_COMPLETED = {}

def show_mode():
    validate_settings(DEVICE, STEPS, USE_REFERENCE)
    print("Mode: INSTRUCTOR REFERENCE; student_* edits are bypassed." if USE_REFERENCE
          else "Mode: STUDENT; saved student_* functions will run.")

show_mode()
print({"device": DEVICE, "steps": STEPS, "output_base": str(OUTPUT_BASE)})


## Level 1 · 2D flow

![Channel and single block](images/chip_2d.png)

Subtract the block $[-1,0]\times[-.5,.1]$ from the channel $[-2.5,2.5]\times[-.5,.5]$. This is a steady incompressible Navier–Stokes problem with $\nu=.02,\rho=1$.
$$u_x+v_y=0,\quad uu_x+vu_y+p_x/\rho-\nu\Delta u=0,\quad uv_x+vv_y+p_y/\rho-\nu\Delta v=0.$$
The inlet has $u=1.5(1-4y^2),v=0$, the outlet has $p=0$, and the walls and block have no-slip conditions. The target flow rate through each vertical fluid cross-section is 1. `fluid_geometry.py` samples the interior outside the block and the exposed walls. Inlet and outlet points are excluded from no-slip samples.
The bundled OpenFOAM data provide an independent numerical comparison for this geometry and flow configuration.

### Code and exercise

Complete the continuity and two momentum residuals in `student_equations` in [chip_2d_l1.py](chip_2d_l1.py). Trace the inlet, outlet, wall, and flow-rate conditions through `loss_terms`, then follow the optimizer loop in `main`. Which terms constrain velocity, and which fix the pressure reference?

### SDF weighting and integrated flow rate

Inside the fluid domain, `sdf_weight` in [fluid_geometry.py](fluid_geometry.py) computes the minimum distance $d>0$ to walls and blocks. The squared continuity and momentum residuals are weighted by $2d$. Points near walls and corners therefore contribute less to this loss; inspect `*_unweighted_rmse` in `metrics.json` to see the unweighted errors as well.

For each vertical cross-section, compute $Q(x,t)=\int_{\text{fluid}}u(x,y,t)\,dy$ and match the inlet flow rate $\int_{-.5}^{.5}1.5(1-4y^2)\,dy=1$. A cross-section that intersects a block is integrated only over its fluid segment. `flux_lines` is the number of cross-sections, and `flux_points` is the number of midpoint quadrature points per cross-section. Compare the roles of the pointwise continuity equation and this integral condition.


In [ ]:
RUN_COMPLETED[1] = False
result_dir = OUTPUT_BASE / f"chip_2d_l1-{uuid4().hex}"
RUN_DIRS[1] = result_dir
command = [sys.executable, str(LAB_DIR / "chip_2d_l1.py"),
           "--steps", str(STEPS), "--seed", str(SEED), "--device", DEVICE,
           "--output-dir", str(result_dir)]
if USE_REFERENCE:
    command.append("--reference")
show_mode()
print("Output:", result_dir)
subprocess.run(command, cwd=LAB_DIR, check=True)
RUN_COMPLETED[1] = True


### Read the results

Inspect the predicted velocity and pressure, then read the `openfoam_*_rmse` comparison with the supplied OpenFOAM fields. Read the PDE, wall, inlet/outlet, and flux errors separately: a good fit to one does not settle the others. The passage above the block has height $0.4$; for $Q=1$, its cross-section mean velocity should be $2.5$. This is a mean, not the maximum speed.

The first and last held-out rows in `loss.csv` use identical points; intermediate rows use freshly sampled training minibatches. `metrics.json` contains the errors, `model.pt` the model and configuration, and `predictions.npz` the fields.


In [ ]:
if not RUN_COMPLETED.get(1, False):
    raise RuntimeError("The current Level 1 training run has not completed.")
result_dir = RUN_DIRS[1]
metrics = show_results(result_dir, steps=STEPS, seed=SEED, reference=USE_REFERENCE)


## Level 2 · Multiple Blocks in Channel

Use the same channel, fluid properties, and inlet/outlet conditions with three blocks.

| Block | x range | y range |
|---|---|---|
| 1 | $[-1,-.4]$ | $[-.5,-.1]$ |
| 2 | $[.2,.7]$ | $[-.5,0]$ |
| 3 | $[1.2,1.6]$ | $[-.5,-.15]$ |

Exclude the block interiors when integrating each cross-section. Inspect the fluid height and midpoint quadrature in `sample_flux`. No independent CFD reference is provided for this level. Interpret the physics residuals and condition errors separately.

### Code and exercise

Complete `student_equations` in [chip_2d_l2.py](chip_2d_l2.py). The governing equations are unchanged from Level 1; identify what the geometry changes in the wall samples and flux integral. Predict where the narrower passages will accelerate the flow.

Look for wakes and their interaction between blocks. Use velocity vectors or regions with $u<0$ to identify possible recirculation; a color map alone does not verify a vortex.


In [ ]:
RUN_COMPLETED[2] = False
result_dir = OUTPUT_BASE / f"chip_2d_l2-{uuid4().hex}"
RUN_DIRS[2] = result_dir
command = [sys.executable, str(LAB_DIR / "chip_2d_l2.py"),
           "--steps", str(STEPS), "--seed", str(SEED), "--device", DEVICE,
           "--output-dir", str(result_dir)]
if USE_REFERENCE:
    command.append("--reference")
show_mode()
print("Output:", result_dir)
subprocess.run(command, cwd=LAB_DIR, check=True)
RUN_COMPLETED[2] = True


### Inspect the flow between blocks

Check whether the predicted acceleration and wake locations match your expectation. Use the integral-continuity error to assess total flow rate, and the unweighted residuals to avoid hiding errors near walls. The Level 1 CFD data are not a reference for this geometry. When comparing learning rates or training duration, keep the sample settings and seed fixed so the held-out points remain the same.


In [ ]:
if not RUN_COMPLETED.get(2, False):
    raise RuntimeError("The current Level 2 training run has not completed.")
result_dir = RUN_DIRS[2]
metrics = show_results(result_dir, steps=STEPS, seed=SEED, reference=USE_REFERENCE)


## Level 3 · Time-Dependent Flow

Return to the single block from Level 1, keep $\rho=1$, and use time interval $[0,10]$ with viscosity $\nu=.01$ (Levels 1 and 2 use $.02$). The inlet and flow rate now start smoothly from rest.
$$u_t+uu_x+vu_y+p_x/\rho-\nu\Delta u=0,\quad v_t+uv_x+vv_y+p_y/\rho-\nu\Delta v=0.$$
$$R(t)=1-e^{-(t/\tau)^2},\qquad u_{\mathrm{in}}(y,t)=1.5(1-4y^2)R(t),\qquad Q(x,t)=R(t).$$
The initial values are $u=v=p=0$. Since $R(0)=R'(0)=0$, the inlet and flux begin at zero with zero initial acceleration, consistent with this initial pressure. The ramp timescale is $\tau=1$, set by `physics.inlet_ramp_time` in [config_chip_2d.yaml](conf/config_chip_2d.yaml); it must be positive and finite. Keep it fixed when comparing attempts.

The outlet still has $p=0$, and walls and the block are no-slip. Flux integration uses the same time at every point of each cross-section and the same ramp as the inlet. This replaces the instantaneous inlet/flux jump in earlier material.
The inputs are $(x,y,t)$ and the outputs are $(u,v,p)$. No independent unsteady CFD reference is supplied.

### Code and exercise

Complete `student_equations` in [chip_2d_l3.py](chip_2d_l3.py) by adding the two time derivatives. `PhysicsInformer` supplies spatial derivatives; the training code supplies time derivatives through PyTorch autograd. Trace `startup_ramp` into both the inlet and integral conditions. Why would ramping the inlet but keeping the target flux at 1 contradict the rest initial condition?

### Temporal evolution and vortex interpretation

Check the initial, inlet, flux, and unweighted PDE errors before interpreting the wake. With the late-time inlet peak $U=1.5$ and block height $L=.6$, $Re=UL/\nu=90$; state the velocity and length scales when quoting a Reynolds number.

As an extension, inspect vorticity $\omega=v_x-u_y$ and time series in the wake. A single snapshot cannot establish vortex shedding or a period. A circular-cylinder Strouhal number is not a reference answer for this wall-attached block; that would require additional time-series inference and independent CFD comparison.


In [ ]:
RUN_COMPLETED[3] = False
result_dir = OUTPUT_BASE / f"chip_2d_l3-{uuid4().hex}"
RUN_DIRS[3] = result_dir
command = [sys.executable, str(LAB_DIR / "chip_2d_l3.py"),
           "--steps", str(STEPS), "--seed", str(SEED), "--device", DEVICE,
           "--output-dir", str(result_dir)]
if USE_REFERENCE:
    command.append("--reference")
show_mode()
print("Output:", result_dir)
subprocess.run(command, cwd=LAB_DIR, check=True)
RUN_COMPLETED[3] = True


### Read the time-dependent result

Compare the initial-condition error with the inlet, flux, and PDE errors, especially near startup. The preview is one mid-time spatial slice; it cannot verify the complete startup history. Compared with Level 1, both time dependence and viscosity changed. Compared with Level 2, the geometry also changed from three blocks to one.


In [ ]:
if not RUN_COMPLETED.get(3, False):
    raise RuntimeError("The current Level 3 training run has not completed.")
result_dir = RUN_DIRS[3]
metrics = show_results(result_dir, steps=STEPS, seed=SEED, reference=USE_REFERENCE)


## Check your understanding

- Why use both pointwise continuity and an integrated flow-rate condition?
- How can wall-distance weighting hide a large near-wall residual in the weighted loss?
- Change either the learning rate or `STEPS`. Keep the seed, sample counts, geometry, PDE, inlet/startup conditions, and evaluation code fixed, then compare the held-out errors and flow field.
- Explore spatial or flux sampling separately. Changing these settings also changes evaluation points or quadrature here; changing condition weights can change the reported loss scale. Treat those runs as exploration, not a like-for-like comparison of held-out values.
- What additional observations would you need before assigning a vortex-shedding frequency?

Next: coupled temperature fields in [Challenge 3 · Climate](../03_climate/Challenge_3_Climate_Modeling.ipynb). [Start Here](../../Start_Here.ipynb).

Adapted from the OpenHackathons materials. [License](../../LICENSE).


--- 

Further resources: [Open Hackathons Resources](https://www.openhackathons.org/s/technical-resources). Community support: [OpenACC and Hackathons Slack Channel](https://www.openacc.org/community#slack).

---

# Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.